# Does the funnel starve the blocks behind it? — a one-cell check on Evo 2 7B

**No GPU needed.** Use a free Colab **CPU** runtime (Runtime → Change runtime type → CPU). The cell
downloads `evo2_7b.pt` (~14 GB), reads the weights, and never builds the model.

## What this is testing, in plain terms

Evo 2 has a strange property we found earlier: **one block near the end writes something about 100,000×
bigger than every other block.** The model stores its running total in 16-bit numbers, which keep only
about 3 significant digits, so once that huge number is in the total, everything written *afterwards*
is too small to change it — like adding a penny to a billion-pound balance. We call it the funnel.

That raises a question about training. If a block's contribution is always rounded away, it never
affects the model's error, so it never receives a correction signal. Standard training also shrinks
unused weights a little at every step ("weight decay"). Over a long training run, a block that is never
corrected but always shrunk should end up with **almost no weights left**.

We have indirect evidence that this happens. In the 7B, the final block writes *exactly the same thing*
on three completely different stretches of DNA — it has stopped responding to its input at all. In the
40B, the 26 blocks after the funnel have weights about **1,000× smaller than they started**.

This cell checks the weights directly, in the one model where the answer is unambiguous.

## The prediction, written down before running (see `PREDICTIONS.md`)

The 7B has five attention blocks: **3, 10, 17, 24, 31**. Only block 31 sits after the funnel (block 30).

> **Predicted:** block 31's attention weights and block 30's MLP weights are **100–1000× smaller** than
> the same weights in blocks 3, 10, 17 and 24, while every block before 30 is at normal scale.
>
> **Refuted if** block 31's attention weights are within **10×** of the other attention blocks — the
> constant write would then need some other explanation.

Because four of the five attention blocks are healthy, the comparison is unambiguous: each block is
compared only against *the same weight tensor in other blocks*, so it is like-for-like.


In [ ]:
# One cell. CPU runtime is fine. ~14 GB download the first time, then cached.
import io, os, re, math, json, torch
from collections import defaultdict

MODEL = 'evo2_7b'
# Reuse a Drive copy if you cached weights there in an earlier round; else download.
CANDIDATES = [f'/content/drive/MyDrive/hf_cache/{MODEL}.pt', f'/content/{MODEL}.pt']
path = next((p for p in CANDIDATES if os.path.exists(p)), None)
if path is None:
    from huggingface_hub import hf_hub_download
    print('downloading evo2_7b.pt (~14 GB, once) ...')
    path = hf_hub_download(repo_id=f'arcinstitute/{MODEL}', filename=f'{MODEL}.pt')
print('checkpoint:', path, f'({os.path.getsize(path)/1e9:.1f} GB)')

# Evo 2 checkpoints carry Transformer Engine `_extra_state` blobs (io.BytesIO, FP8 metadata).
# Allowlisting io.BytesIO keeps weights_only=True protection: BytesIO holds bytes, runs nothing.
try:
    with torch.serialization.safe_globals([io.BytesIO]):
        sd = torch.load(path, map_location='cpu', mmap=True, weights_only=True)
except AttributeError:                      # torch < 2.5 has no safe_globals context manager
    torch.serialization.add_safe_globals([io.BytesIO])
    sd = torch.load(path, map_location='cpu', mmap=True, weights_only=True)
while isinstance(sd, dict) and len(sd) <= 3 and any(k in sd for k in ('state_dict','model','module')):
    sd = sd.get('state_dict') or sd.get('model') or sd.get('module')
n_blob = sum(1 for v in sd.values() if not isinstance(v, torch.Tensor))
sd = {k: v for k, v in sd.items() if isinstance(v, torch.Tensor)}
print(f'{len(sd)} tensors (dropped {n_blob} FP8 metadata blobs)')

# |mean| of every block tensor, grouped BY NAME so each comparison is like-for-like.
per_name = defaultdict(dict)
for k, v in sd.items():
    m = re.match(r'(?:.*\.)?blocks\.(\d+)\.(.+)$', k)
    if m and v.is_floating_point() and v.numel() > 1024:
        per_name[m.group(2)][int(m.group(1))] = float(v.float().abs().mean())
n_blocks = max(b for d in per_name.values() for b in d) + 1
INIT = (1/math.sqrt(8192)) * math.sqrt(2/math.pi)      # |mean| at 1/sqrt(fan_in) initialisation
print(f'{n_blocks} blocks | initialisation |mean| for an 8192-wide layer ~ {INIT:.3e}\n')

FUNNEL, LAST = 30, n_blocks - 1
rows, verdict = [], {}
print(f"{'tensor':34s} {'blks':>4s} {'max':>10s} {'B30 mlp/B31':>12s} {'ratio':>9s}")
for name, per in sorted(per_name.items()):
    if len(per) < 3: continue
    top = max(per.values())
    # the two places the prediction is about
    tgt = per.get(LAST) if 'mha' in name or 'attn' in name else per.get(FUNNEL)
    where = f'b{LAST}' if ('mha' in name or 'attn' in name) else f'b{FUNNEL}'
    ratio = (top / tgt) if tgt else None
    rows.append((name, len(per), top, tgt, ratio, where))
    print(f"{name[:34]:34s} {len(per):4d} {top:10.3e} "
          f"{(f'{tgt:.3e}' if tgt is not None else '-'):>12s} "
          f"{(f'{ratio:.1f}x' if ratio else '-'):>9s}  {where}")

print('\n' + '='*72)
# Clause 1: block 31's attention weights vs the other attention blocks (3, 10, 17, 24)
att = {n: p for n, p in per_name.items() if 'Wqkv' in n or 'out_proj.weight' in n}
print('CLAUSE 1 -- block 31 attention vs blocks 3/10/17/24, same tensor:')
c1 = []
for n, p in sorted(att.items()):
    others = [v for b, v in p.items() if b != LAST]
    mine = p.get(LAST)
    if mine and others:
        r = (sum(others)/len(others)) / mine
        c1.append(r)
        print(f'   {n[:40]:42s} others ~{sum(others)/len(others):.3e} | b{LAST} {mine:.3e} | {r:7.1f}x smaller')
print('\nCLAUSE 2 -- block 30 MLP vs the other blocks\' MLPs:')
c2 = []
for n in ('mlp.l1.weight', 'mlp.l2.weight', 'mlp.l3.weight'):
    p = per_name.get(n, {})
    others = [v for b, v in p.items() if b < FUNNEL]
    mine = p.get(FUNNEL)
    if mine and others:
        r = (sum(others)/len(others)) / mine
        c2.append(r)
        print(f'   {n:42s} upstream ~{sum(others)/len(others):.3e} | b{FUNNEL} {mine:.3e} | {r:7.1f}x smaller')

print('\n' + '='*72)
if c1:
    worst = min(c1)
    print(f'block {LAST} attention is {worst:.1f}x smaller than its siblings '
          f'({min(c1):.1f}-{max(c1):.1f}x across tensors)')
    if worst >= 100:  print('   -> CONFIRMED (predicted 100-1000x)')
    elif worst <= 10: print('   -> REFUTED (predicted >10x; the constant write needs another explanation)')
    else:             print(f'   -> IN BETWEEN ({worst:.1f}x: above the 10x refutation line, below the 100x prediction)')
if c2:
    print(f'block {FUNNEL} MLP is {min(c2):.1f}x smaller than upstream MLPs')
print('='*72)
json.dump({'model': MODEL, 'n_blocks': n_blocks, 'init_mean': INIT,
           'attention_ratios': c1, 'funnel_mlp_ratios': c2,
           'table': [{'tensor': r[0], 'n': r[1], 'max': r[2], 'target': r[3],
                      'ratio': r[4], 'where': r[5]} for r in rows]},
          open('weight_check_7b.json','w'), indent=1)
print('\nwrote weight_check_7b.json -- send this file and the output above')
